# CLIFFGUARD - round 4: the whole ladder under the corrected scorer

Use a T4 GPU, then Run all. Budget **2 hours 30 minutes or more**: about 50
minutes auditing the scorer, then about 1 h 40 re-grading the ladder with it.
Less if a previous session's Drive state restores, since every rung already
graded under the corrected scorer is a cache hit.

The order is deliberate. The audit asks whether the corrected scorer's answer
depends on which letter carries which class --- and the re-grade spends an hour
and forty minutes measuring with that scorer. Learning the answer first costs
nothing and can change what the rest of the session means.

Round 3 established that the original label scorer was comparing
incommensurable logits, and re-graded enough of the ladder to test the headline
4.5-bit comparison: full precision and the 4.5-bit rung on the two behavioural
runs, full precision on the three labelled ones. That is 2 of 8 rungs and 1 of
8.

Every quantity defined over the *whole* ladder is therefore still an
original-scorer quantity: the drift coefficient and its bootstrap, the 14-cell
transition table, the simultaneous one-sided bound, the refusal-law figure, and
all 21 labelled model-by-rung cells.

This notebook grades the remaining rungs. It generates nothing. Every
completion it reads is already in the repository, so there is no corpus to
rebuild, no model under test to download, and nothing to lose if the session
dies -- each grading checkpoints to Drive as it finishes.

**15,200 judge pairs.** 8,000 three-way (2 models x 8 rungs x 500 prompts) and
7,200 five-way (3 models x 8 rungs x 300 prompts). The only throughput this
project has measured is 178 pairs/minute, on a T4 at batch **8**; every timing
here is derived from it, and this session grades at batch **4**, which is
slower. Read the estimates as floors. Batch 4 and not 8 because round 3 graded
at 4, and a ladder graded at two batch sizes is graded by two instruments --- see
`JUDGE_BATCH`. Judge loads are a real line item on top: each grading is a
separate subprocess that reloads the 7B judge, about two minutes each.

Any rung round 3 already graded is a cache hit and costs nothing, so a session
that restores its Drive state grades fewer than 15,200 pairs.


## Environment

The setup is deliberately the same as round 2. It mounts Drive before any model work, so completed schemes survive a Colab disconnect.

In [ ]:
# One flag for "is this session worth spending GPU time on", set here and
# updated by the preflight below. Whether `raise SystemExit` ends a Colab
# "Run all" is a property of the runtime, and a notebook that runs unattended
# for hours should not depend on the answer -- so every cell that spends GPU
# time opens by checking this, and the raises below are belt to its braces.
SESSION_OK = True

import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/parnish007/CLIFFGUARD.git'
# Set to the commit this notebook was validated against; '' follows main.
REPO_COMMIT = '59dfdaa19454e46bf50e74dea70ad930be4b7fd0'
REPO_DIR = pathlib.Path('/content/CLIFFGUARD') if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/cliffguard')

if IN_COLAB:
    # Fatal, not a warning. Without Drive every cache and run directory lives
    # on disk Colab wipes at disconnect, so an unattended session that loses
    # its connection at hour two loses the whole session. Continuing without it
    # is not a degraded run, it is a run that cannot survive the thing most
    # likely to happen to it.
    from google.colab import drive as _drive
    _drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    # Pinned to the commit this notebook was written against. A clone of
    # whatever main happens to be would silently run different analysis code,
    # and --depth 1 cannot reach a specific commit, hence the full clone above.
    if REPO_COMMIT:
        subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], check=True)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                          text=True, check=True).stdout.strip()
    print(f'repo commit  : {head}')
    if REPO_COMMIT and not head.startswith(REPO_COMMIT):
        SESSION_OK = False
        raise SystemExit(f'checked out {head}, expected {REPO_COMMIT}')

    # check=True: a missing bitsandbytes surfaces as a CUDA error inside the
    # judge two hours from now rather than here.
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'bitsandbytes', 'datasets', 'accelerate'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch, numpy as np, transformers
HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else 'NONE'
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0
print(f'repo         : {pathlib.Path.cwd()}')
print(f'python       : {platform.python_version()}')
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {np.__version__}')
print(f'GPU          : {GPU_NAME}  ({VRAM_GB} GB)')
if hasattr(os, 'statvfs'):
    st = os.statvfs('.')
    print(f'free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB')
if not HAS_GPU:
    SESSION_OK = False
    raise SystemExit('No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.')
if tuple(int(p) for p in transformers.__version__.split('.')[:2]) < (4, 45):
    SESSION_OK = False
    raise SystemExit(f'transformers {transformers.__version__} too old (need >= 4.45).\nRun: !pip -q install -U transformers, then restart the runtime.')


## Restore

Drive state first, so the preflight below validates what will actually be used.

In [ ]:
# Restore Drive state BEFORE validating it. On a fresh Colab clone `data/` does
# not exist -- it is gitignored -- so the corpora can only come from Drive or a
# rebuild, and the run directories from a previous session likewise. Validating
# first would fail on files that were about to arrive, or pass a check on files
# that were never going to.
import shutil

def _restore(src: pathlib.Path, dst: pathlib.Path, what: str) -> bool:
    if not src.exists():
        print(f'[drive] no {what} to restore')
        return False
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[drive] restored {what} -> {dst}')
    return True

_restore(DRIVE_ROOT / 'fold_a', pathlib.Path('data/folds/fold_a'), 'Fold A corpus')
_restore(DRIVE_ROOT / 'eval_suites', pathlib.Path('data/eval_suites'), 'eval suites')
_restore(DRIVE_ROOT / 'artifacts' / 'runs', pathlib.Path('artifacts/runs'),
         'previous run directories')

# The five published runs now ship WITH the repository, so the clone already
# has them and step 3 needs no upload. Verified rather than assumed: their
# absence would make the re-grade skip silently, and that is the step whose
# whole purpose is to check the published numbers.
_published = ['colab-behavioural-qwen3b', 'colab-behavioural-phi35',
              'lab-qwen3b-xstest', 'lab-phi35-xstest', 'lab-smol17-xstest']
_found = [p for p in _published
          if list(pathlib.Path('artifacts/runs').glob(f'*{p}'))]
print(f'[repo] published runs available: {len(_found)}/{len(_published)}')
if len(_found) < len(_published):
    print(f'       MISSING {sorted(set(_published) - set(_found))} -- the '
          'step-3 re-grade will skip those and report them as not run')

# A Drive zip still overrides, for anyone re-running with different data --
# but only under artifacts/runs/, which is the only thing a prior-runs archive
# is for.
#
# extractall('.') was unrestricted. CPython does strip '..' and leading
# separators, so nothing escapes the working directory; what it does not stop is
# a member named scripts/classify_completions_judge.py silently replacing the
# grader, or .git/hooks/post-checkout running on the next git command. Both
# were reproduced in a test. The archive comes from the user's own Drive, so
# this is not an attack anyone is mounting -- it is an integrity hole in a
# project whose whole argument is that the instruments were not edited between
# being written and being run, and it costs four lines to close.
_prior = DRIVE_ROOT / 'prior_runs.zip'
if _prior.exists():
    import zipfile
    _allowed, _refused = [], []
    with zipfile.ZipFile(_prior) as zf:
        for _member in zf.namelist():
            _parts = [p for p in _member.replace(chr(92), '/').split('/')
                      if p not in ('', '.', '..')]
            (_allowed if _parts[:2] == ['artifacts', 'runs']
             else _refused).append(_member)
        zf.extractall('.', members=_allowed)
    print(f'[drive] {_prior.name}: unpacked {len(_allowed)} file(s) under '
          f'artifacts/runs/')
    if _refused:
        print(f'        REFUSED {len(_refused)} member(s) outside it, which a '
              'prior-runs archive has no business carrying:')
        for _member in _refused[:10]:
            print(f'          {_member}')
        if len(_refused) > 10:
            print(f'          ... and {len(_refused) - 10} more')

# Rebuild only what is still missing. Fold A comes from an UNPINNED HuggingFace
# revision, so a rebuild can silently produce a different corpus and break the
# pairing with the 48-token runs; preflight hashes it immediately afterwards,
# which is what turns that risk into a caught error rather than a wasted
# session. XSTest is a single file at a stable URL and is safer to fetch.
if not pathlib.Path('data/folds/fold_a/anthropic_hh_refused.jsonl').exists():
    print('[corpus] Fold A missing; rebuilding (preflight will verify the hash)')
    subprocess.run([sys.executable, 'scripts/download_fold_a.py', '--download'],
                   check=True)
if not pathlib.Path('data/eval_suites/xstest.jsonl').exists():
    print('[corpus] XSTest missing; fetching')
    subprocess.run([sys.executable, 'scripts/download_eval_suites.py',
                    '--download', '--suites', 'xstest'], check=False)

# Cache the corpora back, so the next session restores instead of re-fetching.
if DRIVE_ROOT.exists():
    for src, name in ((pathlib.Path('data/folds/fold_a'), 'fold_a'),
                      (pathlib.Path('data/eval_suites'), 'eval_suites')):
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / name, dirs_exist_ok=True)
    print('[drive] corpora cached')

## Preflight

This gate runs before any expensive model load. Do not start the measurements if it reports a failure.

In [ ]:
# --require-xstest because this notebook's second half needs the labelled
# corpus. Without the flag a missing file is only a `skip`, preflight passes,
# and the failure surfaces after the two expensive HH-RLHF steps have already
# spent their GPU time.
preflight = subprocess.run([sys.executable, 'scripts/preflight_round2.py',
                            '--require-xstest'])
if preflight.returncode != 0:
    SESSION_OK = False
    print('PREFLIGHT FAILED. Repair the reported gate failure before running '
          'anything below. Every check above runs on CPU in seconds, so fixing '
          'it costs nothing compared with discovering it at hour three.')

In [ ]:
MODELS_LONG = [('qwen3b', 'Qwen/Qwen2.5-3B-Instruct'), ('phi35', 'microsoft/Phi-3.5-mini-instruct')]
MODELS_XSTEST = MODELS_LONG + [('smollm17b', 'HuggingFaceTB/SmolLM2-1.7B-Instruct')]
JUDGE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
N_LONG, N_XSTEST, LONG_TOKENS, SEED = 250, 150, 256, 0
JUDGE_COMPLETION_CHARS, TAXONOMY_MAX_LENGTH = 2000, 2560
# 4, because that is what round 3 ran, and this notebook exists to EXTEND that
# grading rather than to start a second one. Batch size is not a throughput
# knob here. fp16 kernels reduce in a batch-dependent order, so a pair of label
# logits separated by less than that error can cross, and two batch sizes are
# two instruments.
#
# The two graders punish getting this wrong in opposite ways, and the quieter
# one is worse. The taxonomy grader hashes the batch size into its cache
# fingerprint: a mismatch simply misses every existing cache and recomputes,
# visibly. The three-way judge does not: a mismatch would write the six
# remaining rungs into the SAME cache namespace as round 3's FP16 and 4.5-bit
# rungs, and nothing downstream could tell that the ladder was graded by two
# instruments.
JUDGE_BATCH = 4
# Five verified single-token options rather than label-word prefixes.
# Under Qwen2.5 the old mode compared ' REF', ' COM', ' DEF', ' UNC'
# -- three-character prefixes shared with common words -- against
# ' DISCLAIM', an entire word. The five logits were therefore not
# commensurable, and DISCLAIM, the rarest observed class, was the one
# scored differently from the other four.
SCORING = 'letter'

# Caches go straight to Drive when it is mounted. The old mirror-between-arms
# design left completed schemes on Colab's disposable disk until a whole model
# had finished, so a disconnect could lose hours of valid cache entries.
CACHE_ROOT = (DRIVE_ROOT / 'artifacts') if DRIVE_ROOT.exists() else pathlib.Path('artifacts')
BEHAV_CACHE = str(CACHE_ROOT / 'behavioural_cache')
RESULTS = {}
# Verified by preflight; repeated here so the resume path can reject a
# restored directory whose prompts are not the ones we are pairing against.
FOLD_A_SHA = '7da25bf88ee0409ce4900a12052e15849a2898ed01cfdcdfe6409bbfc11bd9b5'
XSTEST_SHA = '33874ac77bd574a74283cd024466f442e69da870fa1195fcde8a9107433f9ce4'
print(f'long HH-RLHF : {N_LONG} per class, {LONG_TOKENS} tokens, FP16 + RTN 4-bit')
print(f'XSTest       : {N_XSTEST} per class, {LONG_TOKENS} tokens, FP16 only')
print(f'caches       : {CACHE_ROOT}' + ('' if DRIVE_ROOT.exists() else '   (LOCAL -- a disconnect loses them)'))

def run_step(label, script, args, timeout=10800):
    '''Stream one script invocation; keep its tail and exit status.

    The timeout is a watchdog rather than proc.wait(timeout=...). Reading a
    child's stdout to EOF blocks for as long as it lives, so wait was reached
    only after exit and could never stop a hung step. A watchdog kill is kept
    separate because Linux reports it as -9, the same code as an OOM kill.
    '''
    import threading
    cmd = [sys.executable, f'scripts/{script}'] + args
    print(f'\n$ {" ".join(cmd)}', flush=True)
    started, lines = time.time(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timed_out = []
    def _kill():
        timed_out.append(True)
        print(f'\n[{label}] no exit after {timeout / 3600:.1f} h; killing', flush=True)
        proc.kill()
    watchdog = threading.Timer(timeout, _kill)
    watchdog.daemon = True
    watchdog.start()
    try:
        for line in proc.stdout:
            if 'Loading weights' in line or 'it/s]' in line or 's/prompt' in line:
                continue
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait()
    except Exception as exc:
        proc.kill()
        proc.wait()
        lines.append(f'ABORTED: {type(exc).__name__}: {exc}')
    finally:
        watchdog.cancel()
    ok = proc.returncode == 0
    RESULTS[label] = {'returncode': proc.returncode, 'timed_out': bool(timed_out), 'minutes': (time.time() - started) / 60, 'tail': lines[-40:]}
    print(f'\n=== {label}: {"OK" if ok else f"FAILED rc={proc.returncode}"} in {RESULTS[label]["minutes"]:.1f} min ===', flush=True)
    return ok

def run_step_resumable(label, script, args, attempts=3, timeout=10800):
    '''Retry only a real OOM. Each completed scheme is cached immediately, so a
    fresh process resumes farther through the ladder instead of starting over.
    A bad flag, missing checkpoint, or timeout cannot be improved by retrying.'''
    tag = label
    for attempt in range(1, attempts + 1):
        tag = label if attempt == 1 else f'{label}-retry{attempt}'
        if attempt > 1:
            print(f'\n[retry {attempt}/{attempts}] {label}: resuming from cache', flush=True)
        if run_step(tag, script, args, timeout=timeout):
            RESULTS[label] = RESULTS[tag]
            return True
        result = RESULTS[tag]
        if result['returncode'] != -9 or result['timed_out']:
            reason = 'timed out' if result['timed_out'] else f'rc={result["returncode"]}'
            print(f'[{label}] {reason} is not a resumable OOM; not retrying', flush=True)
            RESULTS[label] = result
            return False
    print(f'[{label}] still failing after {attempts} attempts', flush=True)
    RESULTS[label] = RESULTS[tag]
    return False

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # GPU allocation is released between schemes; host memory ratchets upward
    # across model loads and is what eventually triggers Colab's OOM killer.
    host = ''
    try:
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                host = f', host available {float(line.split()[1]) / 1e6:.1f} GB'
    except OSError:
        pass
    print(f'[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated{host}')

def checkpoint_to_drive():
    '''Mirror completed run directories to Drive; caches already live there.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ('runs', 'behavioural_cache', 'sector_cache'):
        src = pathlib.Path('artifacts') / name
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / 'artifacts' / name, dirs_exist_ok=True)
    print(f'[drive] mirrored artifacts/ to {DRIVE_ROOT}')

def restore_from_drive():
    '''Bring back prior run directories before anything runs.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    src = DRIVE_ROOT / 'artifacts' / 'runs'
    if src.exists():
        shutil.copytree(src, pathlib.Path('artifacts') / 'runs', dirs_exist_ok=True)
        print('[drive] restored artifacts/runs')


def latest_run(pattern):
    hits = sorted(pathlib.Path('artifacts/runs').glob(pattern))
    return hits[-1] if hits else None

def completed_run(label, schemes, model, n_prompts, corpus_sha=None,
                  tokens=None):
    '''A restored run directory that is genuinely finished, not merely present.

    Existence is not completion. A Drive copy interrupted mid-write, a stale
    directory from a run with different arguments, or a truncated JSON all look
    identical to `path.exists()`, and treating any of them as done would skip
    the step and hand the analysis silently wrong data. That is worse than
    re-running: a missing result is visible, a wrong one is not.

    So everything the step depends on is checked against the manifest --
    model, scheme list, prompt count, token budget, seed, and the ordered
    corpus hash that makes the comparison paired at all -- and every
    completions file is parsed and counted rather than stat-ed.
    '''
    tokens = LONG_TOKENS if tokens is None else tokens
    complete = []
    for run in sorted(pathlib.Path('artifacts/runs').glob(f'*_{label}')):
        try:
            manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
        except (OSError, ValueError):
            print(f'[resume] {run.name}: unreadable manifest; ignoring')
            continue
        expected = {'model_id': model, 'n_prompts': n_prompts,
                    'max_new_tokens': tokens, 'seed': SEED}
        wrong = {k: (manifest.get(k), v) for k, v in expected.items()
                 if manifest.get(k) != v}
        if wrong:
            print(f'[resume] {run.name}: arguments differ {wrong}; ignoring')
            continue
        if list(manifest.get('schemes', [])) != schemes:
            print(f'[resume] {run.name}: schemes {manifest.get("schemes")} '
                  f'!= {schemes}; ignoring')
            continue
        digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
        if corpus_sha and digest != corpus_sha:
            print(f'[resume] {run.name}: corpus hash differs; ignoring')
            continue
        ok = True
        for scheme in schemes:
            path = run / 'results' / f'completions_{scheme}.json'
            try:
                blob = json.loads(path.read_text(encoding='utf-8'))
                texts = blob['completions'] if isinstance(blob, dict) else blob
            except (OSError, ValueError, KeyError):
                print(f'[resume] {run.name}: {path.name} missing or unparseable')
                ok = False
                break
            if len(texts) != n_prompts:
                print(f'[resume] {run.name}: {path.name} has {len(texts)} rows, '
                      f'expected {n_prompts}')
                ok = False
                break
        if ok:
            complete.append(run)
    if len(complete) > 1:
        raise SystemExit(f'multiple completed runs share {label}: '
                         f'{[p.name for p in complete]}. Refusing to choose one.')
    return complete[0] if complete else None


def grade_complete(run, filename, schemes, n_prompts):
    '''A grading output that covers every scheme and every prompt.

    Same argument as above: a half-written grade file exists just as much as a
    finished one, and skipping the grader on the strength of that would leave
    the analysis reading rows that were never produced.
    '''
    path = run / 'results' / filename
    try:
        blob = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return False
    # The scorer is part of what makes a grading the right grading. A file
    # written under first-token scoring satisfies every structural check here
    # while being the instrument this round exists to replace, so skipping on
    # its presence would silently keep the old measurement.
    if blob.get('scoring') not in (None, SCORING):
        print(f"[resume] {path.name}: scored under "
              f"{blob.get('scoring')!r}, not {SCORING!r}; will re-grade")
        return False
    results = blob.get('results') or blob.get('verdicts') or {}
    missing = [s for s in schemes if s not in results]
    if missing:
        print(f'[resume] {path.name}: no rows for {missing}; will re-grade')
        return False
    return True


def print_pairing(run):
    """Show the ordered corpus hash, so pairing is visible not assumed."""
    manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
    digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
    print(f'[run] {run}')
    print(f'[pairing] corpora.prompts.sha256_ordered = {digest}')


def record_skip(label, reason):
    RESULTS[label] = {'returncode': 0, 'skipped': True, 'minutes': 0.0,
                      'tail': [reason]}
    print(f'[{label}] already complete after Drive restore; skipping: {reason}')


## Step 1 --- audit the scorer, before measuring with it

The corrected scorer replaced label words with single-token letters. That is
what makes the five logits commensurable, and it is not the end of the
question: multiple-choice prompting carries position and symbol preferences, so
a judge that leans toward option A --- whatever A happens to say --- would
produce a grading that passes every check the manuscript makes and still be
reading its own prior off the page.

This grades the same stored text again with the five descriptions held **word
for word** and only the letter each is offered under permuted, then maps the
verdicts back to classes. Two permutations per grader, chosen so every class
appears under at least three different letters across the replicates:

| grader | canonical | replicate 1 | replicate 2 |
|---|---|---|---|
| three-way | REFUSE, COMPLY, UNCLEAR | cyclic shift | reverse |
| five-way | REFUSE ... UNCLEAR | shift by two | reverse |

**8,700 pairs, about 50 minutes.** Restricted to the rungs the manuscript
quotes --- full precision and 4.5 bits for the behavioural runs, full precision
for the labelled ones --- because a symbol preference is a property of the judge
and the prompt, not of the bit-width of the text being judged.

That count is 2,900 canonical and 5,800 permuted. The canonical pass is included
because a permutation is only meaningful against it, and a clone that has not
restored a previous session carries only the *original* grading --- the
corrected one is a result, and results are not committed to this repository.

Step 2 reuses 2,000 of those 2,900. The three-way judge's cache fingerprint does
not depend on which schemes were graded together, so the behavioural rungs
graded here are cache hits when step 2 asks for the whole ladder. The five-way
fingerprint hashes the graded *text* of the scheme set it was handed, so an
FP16-only grading and an eight-rung one are different fingerprints: the 900
labelled pairs are graded twice, about five minutes. The alternative is an audit
with nothing to compare against.

**This runs before the re-grade, deliberately.** The re-grade spends an hour and
forty minutes putting the whole ladder on this scorer; whether that scorer is
order-sensitive is worth knowing at minute thirty rather than at minute one
hundred and forty. It does not block the re-grade. A ladder graded under one
fixed assignment is a coherent measurement either way --- what the audit changes
is what may be claimed about it.

Each permutation lands in its own cache: the assignment changes the prompt,
which changes the fingerprint, which is the filename. Nothing here can overwrite
the canonical grading.

In [ ]:
# The plan, outside the guard. These are declarations rather than work --
# they name what the session intends to do -- and later cells read them
# whether or not the steps ran. Left inside, a skipped step took the
# availability readout and the export down with it on a NameError.
# Permutations, written out rather than generated, so the notebook records
# exactly which assignments were run. THREE_WAY and FIVE_WAY are the canonical
# assignments and are not in the lists below; the canonical grading is produced
# further down, before the permutations, because on a fresh clone there is
# nothing on disk to compare a permutation against.
THREE_WAY = ['REFUSE', 'COMPLY', 'UNCLEAR']
FIVE_WAY = ['REFUSE', 'COMPLY', 'DEFLECT', 'DISCLAIM', 'UNCLEAR']

ORDERS_3 = [
    ['COMPLY', 'UNCLEAR', 'REFUSE'],   # cyclic shift: REFUSE at C
    ['UNCLEAR', 'COMPLY', 'REFUSE'],   # reverse
]
ORDERS_5 = [
    ['DEFLECT', 'DISCLAIM', 'UNCLEAR', 'REFUSE', 'COMPLY'],   # shift by two
    ['UNCLEAR', 'DISCLAIM', 'DEFLECT', 'COMPLY', 'REFUSE'],   # reverse
]

# Only the rungs the manuscript quotes. A letter preference is a property of the
# judge and the prompt; it does not become a different property at 3 bits.
ORDER_RUNGS = {'behavioural': ['FP16', 'RTN_4B'], 'labelled': ['FP16']}

PERMUTE = [
    ('qwen3b',        '*colab-behavioural-qwen3b', 'classify_completions_judge.py',
     ORDERS_3, ORDER_RUNGS['behavioural']),
    ('phi35',         '*colab-behavioural-phi35',  'classify_completions_judge.py',
     ORDERS_3, ORDER_RUNGS['behavioural']),
    ('qwen3b-xstest', '*lab-qwen3b-xstest', 'classify_completion_taxonomy.py',
     ORDERS_5, ORDER_RUNGS['labelled']),
    ('phi35-xstest',  '*lab-phi35-xstest',  'classify_completion_taxonomy.py',
     ORDERS_5, ORDER_RUNGS['labelled']),
    ('smol17-xstest', '*lab-smol17-xstest', 'classify_completion_taxonomy.py',
     ORDERS_5, ORDER_RUNGS['labelled']),
]

if not (SESSION_OK):
    print('SKIPPED: preflight failed, so nothing here is run.')
else:
    # Permutations plus the canonical grading each is compared against, hence
    # (orders + 1): a clone that has not restored a previous session carries
    # only the ORIGINAL grading.
    #
    # Of the canonical half, step 2 reuses the behavioural rungs and not the
    # labelled ones. The three-way judge's fingerprint does not depend on which
    # schemes were graded together, so an FP16+4.5-bit grading here is a cache
    # hit when step 2 asks for the full ladder. The five-way fingerprint hashes
    # the graded TEXT of the scheme set it was given, so an FP16-only grading
    # and an eight-rung one are different fingerprints and the 900 labelled
    # pairs below are genuinely graded twice. Five minutes, and the alternative
    # is an audit with nothing to compare against.
    def _pairs(selector):
        return sum(selector(len(orders)) * len(rungs)
                   * (500 if 'xstest' not in tag else 300)
                   for tag, _p, _s, orders, rungs in PERMUTE)

    canonical, permuted = _pairs(lambda _n: 1), _pairs(lambda n: n)
    print(f'option-order audit: {canonical + permuted} judge pairs '
          f'({canonical} canonical, {permuted} permuted) '
          f'~{(canonical + permuted) / 178 / 60:.1f} h at the 178 pairs/min '
          'round 3 measured at batch 8; this session grades at batch 4, which '
          'is slower, so read it as a floor')

    for tag, pattern, script, orders, rungs in PERMUTE:
        run_dir = latest_run(pattern)
        if run_dir is None:
            print(f'[{tag}] run not found; skipping')
            continue
        present = [s for s in rungs
                   if (run_dir / 'results' / f'completions_{s}.json').exists()]
        if not present:
            print(f'[{tag}] none of {rungs} present; skipping')
            continue
        # The canonical grading first, on exactly the rungs this audit compares.
        # A permutation is compared against it, and a clone that has not restored a
        # previous session's Drive state carries only the ORIGINAL grading -- the
        # corrected one is a result, and results are not committed here. Without this
        # the audit grades its permutations, finds nothing to compare them with, and
        # reports itself incomplete after spending the GPU time.
        #
        # Mostly moved work rather than extra: step 2 cache-hits the behavioural
        # rungs, whose fingerprint is independent of the scheme set. The three
        # labelled rungs it does not, for the reason given above the pair count.
        canonical_label = f'r4-canonical-{tag}'
        if canonical_label not in RESULTS:
            free_vram()
            canonical_args = [
                str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
                '--completion-chars', str(JUDGE_COMPLETION_CHARS),
                '--batch-size', str(JUDGE_BATCH), '--scoring', 'letter',
                '--schemes', *present]
            if script == 'classify_completion_taxonomy.py':
                canonical_args += ['--max-length', str(TAXONOMY_MAX_LENGTH)]
            print()
            print(f'{canonical_label}: the canonical assignment, {len(present)} '
                  'rung(s), so the permutations have something to compare against')
            run_step_resumable(canonical_label, script, canonical_args,
                               timeout=45 * 60)
            checkpoint_to_drive()

        for index, order in enumerate(orders, start=1):
            label = f'r4-order{index}-{tag}'
            free_vram()
            args = [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
                    '--completion-chars', str(JUDGE_COMPLETION_CHARS),
                    '--batch-size', str(JUDGE_BATCH), '--scoring', 'letter',
                    '--letter-order', ','.join(order),
                    '--schemes', *present]
            if script == 'classify_completion_taxonomy.py':
                args += ['--max-length', str(TAXONOMY_MAX_LENGTH)]
            print(f'\n{label}: {" ".join(order)}')
            run_step_resumable(label, script, args, timeout=45 * 60)
            checkpoint_to_drive()

    print('\noption-order replicates finished')

In [ ]:
if not (SESSION_OK):
    print('SKIPPED: preflight failed, so nothing here is run.')
else:
    # Read the audit here rather than at home. This is the number that decides what
    # the rest of the session is worth, and it costs nothing to look at it now.

    # Which gradings were supposed to happen, counted from the plan rather than from
    # whatever files are on disk. A summary over an unknown fraction of the audit
    # reads exactly like a summary over all of it, and the failure it hides -- some
    # permutations OOMed, the rest agreed -- is the one that would produce a
    # reassuring number from a broken experiment.
    expected_files, missing_jobs = {}, []
    for tag, pattern, script, orders, rungs in PERMUTE:
        for index in range(1, len(orders) + 1):
            label = f'r4-order{index}-{tag}'
            result = RESULTS.get(label)
            if result is None or result.get('returncode') not in (0, None):
                missing_jobs.append(label)
        expected_files[tag] = len(orders)

    for tag, pattern, script, orders, rungs in PERMUTE:
        run_dir = latest_run(pattern)
        if run_dir is None:
            continue
        out_path = f'artifacts/runs/letter_order_{tag}.json'
        # Removed first, so a file from an earlier attempt cannot be read as this
        # session's result when the analysis below fails to overwrite it.
        pathlib.Path(out_path).unlink(missing_ok=True)
        cmd = [sys.executable, 'scripts/analyse_letter_order.py', str(run_dir)]
        for order in orders:
            cmd += ['--orders', ','.join(order)]
        if script == 'classify_completion_taxonomy.py':
            cmd.append('--five-way')
        cmd += ['--batch-size', str(JUDGE_BATCH), '--out', out_path]
        if subprocess.run(cmd).returncode != 0:
            missing_jobs.append(f'analyse-{tag}')

    # One verdict over every permutation of every run, in two numbers.
    #
    # Class agreement is the headline: how often the permuted grading gives the same
    # CLASS as the canonical one. Letter agreement is the diagnostic beside it. A
    # judge with no symbol preference produces DIFFERENT letter frequencies under a
    # permutation, because the classes moved and the letters followed; a judge that
    # answers "A" regardless produces the SAME letter frequency and different
    # classes. High class agreement with moved letters is the good case. Low class
    # agreement with frozen letters is the pathology.
    ORDER_AUDIT = {'rows': [], 'worst_class_agreement': None,
                   'letter_frequency_moved': None}
    # Named, not globbed. A glob would also pick up a file for a run this session
    # never touched.
    for tag in expected_files:
        path = pathlib.Path(f'artifacts/runs/letter_order_{tag}.json')
        if not path.exists():
            missing_jobs.append(f'result-{tag}')
            continue
        blob = json.loads(path.read_text(encoding='utf-8'))
        # What arrived, against what was asked for. A canonical grading that
        # covers full precision and not the 4.5-bit rung produces an analysis
        # that exits 0 and writes a file with fewer comparisons than planned,
        # so checking the exit status alone calls that complete.
        got_orders = {order for block in blob['runs'].values()
                      for order in block['orders']}
        if len(got_orders) < expected_files[tag]:
            missing_jobs.append(
                f'comparisons-{tag} ({len(got_orders)} of '
                f'{expected_files[tag]} permutations produced one)')
        expected_rungs = len(ORDER_RUNGS['labelled' if 'xstest' in tag
                                         else 'behavioural'])
        for run_name, block in blob['runs'].items():
            for order, schemes in block['orders'].items():
                if len(schemes) < expected_rungs:
                    missing_jobs.append(
                        f'rungs-{tag}-{order} ({len(schemes)} of '
                        f'{expected_rungs})')
                for scheme, row in schemes.items():
                    ORDER_AUDIT['rows'].append({
                        'run': run_name, 'order': order, 'scheme': scheme,
                        'n': row['n'], 'agreement': row['agreement'],
                        'letters_moved': row['letters']['canonical'] != row['letters']['permuted'],
                    })

    ORDER_AUDIT['missing'] = sorted(set(missing_jobs))
    ORDER_AUDIT['complete'] = not ORDER_AUDIT['missing']

    if ORDER_AUDIT['missing']:
        print(f"\n{'=' * 68}")
        print('OPTION-ORDER AUDIT INCOMPLETE')
        print(f"  {len(ORDER_AUDIT['missing'])} job(s) did not produce a result:")
        for name in ORDER_AUDIT['missing']:
            print(f'    {name}')
        print('  Whatever the rows below say, they are a summary of PART of the')
        print('  audit and cannot establish that the scorer is order-insensitive.')
        print('  The re-grade in step 2 still runs: a ladder under one fixed')
        print('  assignment is a coherent measurement either way.')
        print('=' * 68)

    if ORDER_AUDIT['rows']:
        agreements = [r['agreement'] for r in ORDER_AUDIT['rows']]
        ORDER_AUDIT['worst_class_agreement'] = min(agreements)
        ORDER_AUDIT['letter_frequency_moved'] = all(r['letters_moved']
                                                    for r in ORDER_AUDIT['rows'])
        worst = ORDER_AUDIT['worst_class_agreement']
        print(f"\n{'=' * 68}")
        print(f'OPTION-ORDER AUDIT: {len(agreements)} comparisons')
        print(f'  worst class agreement : {100 * worst:.1f}%')
        print(f'  mean class agreement  : {100 * sum(agreements) / len(agreements):.1f}%')
        print(f'  letter frequency moved in every comparison: '
              f"{ORDER_AUDIT['letter_frequency_moved']}")
        if not ORDER_AUDIT['complete']:
            print('  -> INCOMPLETE; the numbers above describe only the '
                  'comparisons that ran.')
        elif worst >= 0.95:
            print('  -> the letters carry no detectable signal. The corrected '
                  'scorer measures the completion.')
        elif worst >= 0.85:
            print('  -> a small order effect. Report it; the ladder below is still '
                  'one fixed assignment throughout.')
        else:
            print('  -> A REAL ORDER EFFECT. The re-grade below is still worth '
                  'running, because a ladder under one fixed assignment is a '
                  'coherent measurement -- but it is an assignment-dependent one, '
                  'and the manuscript must say so beside every corrected number.')
        print('=' * 68)
    else:
        print('\nNo option-order results to summarise; the audit did not run.')

## Step 2 --- the re-grade

Every rung, under the corrected scorer, so that the quantities defined over the
whole ladder stop being original-scorer quantities: the drift coefficient and
its bootstrap, the 14-cell transition table, the simultaneous one-sided bound,
the refusal-law figure, and all 21 labelled model-by-rung cells.

It generates nothing. Every completion it reads is already in the repository, so
there is no corpus to rebuild, no model under test to download, and nothing to
lose if the session dies --- each grading checkpoints to Drive as it finishes,
and any rung round 3 already graded is a cache hit that costs nothing.

In [ ]:
# The plan, outside the guard. These are declarations rather than work --
# they name what the session intends to do -- and later cells read them
# whether or not the steps ran. Left inside, a skipped step took the
# availability readout and the export down with it on a NameError.
# Every rung, not just the two that carry the headline. The point of this
# notebook is the quantities that are defined over the whole ladder, and those
# cannot be assembled from a partial re-grade.
ALL_RUNGS = ['FP16', 'RTN_8B', 'RTN_7B', 'RTN_6B', 'RTN_5B', 'RTN_4B',
             'RTN_3B', 'RTN_2B']

REGRADE = [
    ('qwen3b',        '*colab-behavioural-qwen3b', 500, 'classify_completions_judge.py'),
    ('phi35',         '*colab-behavioural-phi35',  500, 'classify_completions_judge.py'),
    ('qwen3b-xstest', '*lab-qwen3b-xstest',        300, 'classify_completion_taxonomy.py'),
    ('phi35-xstest',  '*lab-phi35-xstest',         300, 'classify_completion_taxonomy.py'),
    ('smol17-xstest', '*lab-smol17-xstest',        300, 'classify_completion_taxonomy.py'),
]

if not (SESSION_OK):
    print('SKIPPED: preflight failed, so nothing here is run.')
else:
    missing = []
    for tag, pattern, n_prompts, script in REGRADE:
        run_dir = latest_run(pattern)
        if run_dir is None:
            missing.append(tag)
            continue

        # Grade only the rungs whose completions are actually present. A run that
        # stops short of the full ladder is not a failure, and naming a scheme with
        # no completions file would abort the whole grading rather than skip it.
        present = [s for s in ALL_RUNGS
                   if (run_dir / 'results' / f'completions_{s}.json').exists()]
        if not present:
            missing.append(f'{tag} (no completions)')
            continue

        label = f'r4-{tag}'
        free_vram()
        args = [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
                '--completion-chars', str(JUDGE_COMPLETION_CHARS),
                # Held identical to round 3's partial re-grade, so the rungs
                # graded there and the rungs graded here are one measurement.
                # See JUDGE_BATCH above for why this is not a throughput knob.
                '--batch-size', str(JUDGE_BATCH),
                '--scoring', 'letter',
                '--schemes', *present]
        if script == 'classify_completion_taxonomy.py':
            args += ['--max-length', str(TAXONOMY_MAX_LENGTH)]

        print(f'\n{label}: grading {run_dir.name}')
        print(f'  {len(present)} rungs x {n_prompts} prompts = '
              f'{len(present) * n_prompts} pairs under letter scoring')
        run_step_resumable(label, script, args, timeout=60 * 60)
        checkpoint_to_drive()

    if missing:
        print(f'\nNOT RE-GRADED: {missing}')
        print('These runs ship with the repository, so a missing one means the '
              'clone or the Drive restore is incomplete rather than that the data '
              'does not exist. Every ladder-wide quantity stays an original-scorer '
              'quantity until they are graded.')
    else:
        print('\nevery rung of all five published runs graded under the corrected scorer')


## What is now available

A count rather than a claim. The cell below reports, per run, how many rungs
carry a corrected-scorer cache, so the answer to *can the ladder-wide
quantities be recomputed* is a number on the screen and not an inference from
the absence of an error.

In [ ]:
from cliffguard.eval.scorer_caches import resolve, resolve_taxonomy

print(f"{'run':38s} {'scorer':14s} {'rungs':>5s}")
print('-' * 62)
complete = True
for tag, pattern, n_prompts, script in REGRADE:
    run_dir = latest_run(pattern)
    if run_dir is None:
        continue
    if script == 'classify_completions_judge.py':
        found = resolve(run_dir, completion_chars=600)
        prefix = 'judge'
    else:
        # Explicit: resolve_taxonomy recomputes the fingerprint, and the
        # batch size is part of it. Left at the library default this readout
        # would report zero rungs for a grading that had just succeeded.
        found = resolve_taxonomy(run_dir, batch_size=JUDGE_BATCH)
        prefix = 'taxonomy'
    digest = found.get('letter')
    rungs = sorted(p.stem.split('_', 2)[-1]
                   for p in run_dir.glob(f'results/{prefix}_{digest}_*.json')) if digest else []
    print(f'{run_dir.name[-38:]:38s} {"letter":14s} {len(rungs):5d}')
    if len(rungs) < 8:
        complete = False

print()
if complete:
    print('Every published run now has all eight rungs under the corrected '
          'scorer. The drift coefficient, the transition table, the '
          'simultaneous bound and the 21 labelled cells can all be recomputed.')
else:
    print('At least one run is short of the full ladder. Ladder-wide quantities '
          'must stay labelled as original-scorer results until it is not.')


## Export

The archive carries the re-graded run directories. Activations are excluded:
they are 63 to 95 MB each, nothing here reads them, and the probe refit that
does read them runs locally on a CPU.

As in round 3, the filename is prefixed `INCOMPLETE_` if any grading failed or
never ran, because a step that produced nothing is not a step that found
nothing.

In [ ]:
import zipfile

stamp = time.strftime('%Y%m%d-%H%M%S')
failed = [k for k, v in RESULTS.items()
          if v.get('returncode') not in (0, None) or v.get('blocked')]
never_ran = [f'r4-{tag}' for tag, *_ in REGRADE
             if f'r4-{tag}' not in RESULTS]
# Separate, and deliberately NOT part of `never_ran`. The re-grade is what the
# ladder-wide quantities need; the option-order replicates are an additional
# check on the scorer. An archive that carries the first and not the second is
# complete for what it is for, and marking it INCOMPLETE would say otherwise.
# globals(), because this cell has to work in a session where the option-order
# cell was skipped, interrupted, or never reached. Referring to PERMUTE directly
# would turn "the GPU died during the replicates" into a NameError that also
# loses the completed re-grade, which is the one thing in the archive that
# cannot be recomputed without another two hours.
_permute = globals().get('PERMUTE', [])
orders_missing = [f'r4-order{i}-{tag}'
                  for tag, _p, _s, orders, _r in _permute
                  for i in range(1, len(orders) + 1)
                  if f'r4-order{i}-{tag}' not in RESULTS]
prefix = 'INCOMPLETE_' if (failed or never_ran) else ''
# /content only exists on Colab. Off it the notebook is still runnable --
# that is how its logic gets exercised without a GPU -- and an unconditional
# /content path made the last cell the one step that could not be dry-run.
name = f'{prefix}cliffguard_round4_{stamp}.zip'
archive = pathlib.Path(f'/content/{name}' if IN_COLAB else name)

status = {'steps': RESULTS, 'failed': failed, 'never_ran': never_ran,
          'option_orders_missing': orders_missing,
          'option_orders': {'three_way': globals().get('ORDERS_3'),
                            'five_way': globals().get('ORDERS_5')},
          # The audit's verdict travels with the archive. A reader who has only
          # the zip must be able to see that the scorer was checked before the
          # ladder was graded with it, and what the check said.
          'option_order_audit': globals().get('ORDER_AUDIT'),
          'scoring': 'letter', 'rungs': ALL_RUNGS,
          'completion_chars': JUDGE_COMPLETION_CHARS,
          'taxonomy_max_length': TAXONOMY_MAX_LENGTH,
          'batch_size': JUDGE_BATCH}

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for tag, pattern, n_prompts, script in REGRADE:
        run_dir = latest_run(pattern)
        if run_dir is None:
            continue
        for path in sorted(run_dir.rglob('*')):
            if not path.is_file() or 'activations' in path.parts:
                continue
            # as_posix on the path as found, NOT relative_to(REPO_DIR).
            # `latest_run` globs a relative path, so relative_to() against an
            # absolute REPO_DIR raises -- and it raises HERE, after the whole
            # session has been spent, with the archive already opened and
            # therefore left truncated. Round 3 got this right; round 4 had
            # regressed it.
            zf.write(path, path.as_posix())
    for path in sorted(pathlib.Path('artifacts/runs').glob('letter_order_*.json')):
        zf.write(path, path.as_posix())
    zf.writestr('artifacts/runs/ROUND4_STATUS.json', json.dumps(status, indent=2))

print(f'wrote {archive}  ({archive.stat().st_size / 1e6:.1f} MB)')
if failed or never_ran:
    print(f'INCOMPLETE: failed={failed} never_ran={never_ran}')
else:
    print('every rung re-graded under the corrected scorer')
if not _permute:
    print('option-order replicates: cell not run, so none are in this archive')
elif orders_missing:
    print(f'option-order replicates not run: {orders_missing}')
else:
    print('option-order replicates included')

# To Drive as well: an unattended browser download is not durable.
try:
    shutil.copy2(archive, DRIVE_ROOT / archive.name)
    print(f'copied to Drive: {archive.name}')
except Exception as error:
    print(f'could not copy to Drive ({error}); download it from /content')

try:
    from google.colab import files
    files.download(str(archive))
except Exception:
    pass
